In [4]:
# ============================================================
# 02_keystroke_features.ipynb
# Adaptive Continuous Authentication — Keystroke Feature Builder
# ============================================================

"""
Goal:
  - Load keystroke_raw.parquet (timing columns only)
  - Build rich feature vectors per repetition or session
  - Save compact numpy arrays for model training

Output:
  data/keystroke_features.npz
"""

# ============================================================
# Setup
# ============================================================

import numpy as np
import pandas as pd
import scipy.stats as st
from pathlib import Path

DATA_DIR = Path("data")
RAW_PATH = DATA_DIR / "keystroke_raw.parquet"
OUT_PATH = DATA_DIR / "keystroke_features.npz"

print("📥 Loading raw keystroke timings...")
df = pd.read_parquet(RAW_PATH, engine="fastparquet")
print(f"Loaded {len(df)} rows.")

# Identify timing feature columns
exclude_cols = {"user_id", "session_id", "timestamp"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Detected {len(feature_cols)} raw timing columns.")

# ============================================================
# 1. Helper functions for behavioral features
# ============================================================

def stats_1d(x):
    """Compute robust 1D statistical features."""
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "median": float(np.median(x)),
        "iqr": float(st.iqr(x)),
        "skew": float(st.skew(x)),
        "kurt": float(st.kurtosis(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

def entropy_measure(x, num_bins=20):
    """Entropy of dwell/flight distribution."""
    hist, _ = np.histogram(x, bins=num_bins, density=True)
    hist = hist + 1e-8
    return float(-np.sum(hist * np.log(hist)))

def outlier_rate(x, z_thresh=3):
    """Fraction of >3-sigma timing outliers."""
    if len(x) < 2: 
        return 0.0
    z = np.abs((x - np.mean(x)) / np.std(x))
    return float(np.mean(z > z_thresh))

def extract_ngram_stats(df_row, prefix):
    """Extract stats for features matching H.*, DD.*, or UD.* groups."""
    cols = [c for c in feature_cols if c.startswith(prefix)]
    x = df_row[cols].values.astype(np.float32)
    return stats_1d(x), x


# ============================================================
# 2. Build feature vectors for each row
# ============================================================

rows = []
user_ids = []
session_ids = []

print("🔧 Building feature vectors...")

for idx, row in df.iterrows():
    user_ids.append(int(row["user_id"]))
    session_ids.append(int(row["session_id"]))

    # --- Extract per-group stats ---
    dwell_stats, dwell_raw = extract_ngram_stats(row, prefix="H.")
    dd_stats, dd_raw       = extract_ngram_stats(row, prefix="DD.")
    ud_stats, ud_raw       = extract_ngram_stats(row, prefix="UD.")

    # --- Per-repetition typing speed (keys/sec) ---
    total_keys = len(dwell_raw)
    mean_dwell = np.mean(dwell_raw)
    typing_speed = float(total_keys / (np.sum(dwell_raw) + 1e-6))

    # --- Rhythm variability ---
    rhythm_entropy = entropy_measure(dd_raw)
    hesitation_entropy = entropy_measure(ud_raw)

    # --- Outlier rate ---
    dwell_outliers = outlier_rate(dwell_raw)
    dd_outliers = outlier_rate(dd_raw)
    ud_outliers = outlier_rate(ud_raw)

    # --- Collapse dicts into a flat vector ---
    feats = [
        # dwell stats
        dwell_stats["mean"], dwell_stats["std"], dwell_stats["median"],
        dwell_stats["iqr"], dwell_stats["skew"], dwell_stats["kurt"],
        dwell_stats["min"], dwell_stats["max"],

        # dd stats
        dd_stats["mean"], dd_stats["std"], dd_stats["median"],
        dd_stats["iqr"], dd_stats["skew"], dd_stats["kurt"],
        dd_stats["min"], dd_stats["max"],

        # ud stats
        ud_stats["mean"], ud_stats["std"], ud_stats["median"],
        ud_stats["iqr"], ud_stats["skew"], ud_stats["kurt"],
        ud_stats["min"], ud_stats["max"],

        # behavior signals
        typing_speed,
        rhythm_entropy,
        hesitation_entropy,
        dwell_outliers,
        dd_outliers,
        ud_outliers,
    ]

    rows.append(feats)

X = np.array(rows, dtype=np.float32)
y_user = np.array(user_ids, dtype=np.int32)
y_sess = np.array(session_ids, dtype=np.int32)

print(f"Feature matrix shape: {X.shape}")

# ============================================================
# 3. Save output
# ============================================================

print(f"💾 Saving features → {OUT_PATH}")
np.savez_compressed(
    OUT_PATH,
    features=X,
    user_id=y_user,
    session_id=y_sess,
    feature_cols=np.array(feature_cols, dtype=object),
)

print("✅ Done! Keystroke features ready for encoder training.")

📥 Loading raw keystroke timings...
Loaded 20400 rows.
Detected 31 raw timing columns.
🔧 Building feature vectors...
Feature matrix shape: (20400, 30)
💾 Saving features → data/keystroke_features.npz
✅ Done! Keystroke features ready for encoder training.
